# FarmSight XAI Project Setup
This notebook handles the initial workspace environment setup, including mounting Google Drive, configuring directory paths, and documenting key library dependencies.

In [ ]:
import os
import sys
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# Define paths
base_path = '/content/drive/MyDrive/FarmSight_XAI'
folders_to_create = [
    os.path.join(base_path, 'data/raw'),
    os.path.join(base_path, 'data/interim'),
    os.path.join(base_path, 'data/processed'),
    os.path.join(base_path, 'models'),
    os.path.join(base_path, 'results/figures'),
    os.path.join(base_path, 'results/tables'),
    os.path.join(base_path, 'logs')
]

# 2. & 3. Create folder structure and print every created folder path
print("Creating project directory structure:")
for folder in folders_to_create:
    os.makedirs(folder, exist_ok=True)
    print(f"Created or verified: {folder}")

Mounted at /content/drive
Creating project directory structure:
Created or verified: /content/drive/MyDrive/FarmSight_XAI/data/raw
Created or verified: /content/drive/MyDrive/FarmSight_XAI/data/interim
Created or verified: /content/drive/MyDrive/FarmSight_XAI/data/processed
Created or verified: /content/drive/MyDrive/FarmSight_XAI/models
Created or verified: /content/drive/MyDrive/FarmSight_XAI/results/figures
Created or verified: /content/drive/MyDrive/FarmSight_XAI/results/tables
Created or verified: /content/drive/MyDrive/FarmSight_XAI/logs


In [ ]:
# 4. Print Python version
python_version = sys.version
print(f"Python Version: {python_version}")

# 5. Retrieve currently installed versions of required libraries
libraries = ['pandas', 'numpy', 'scikit-learn', 'matplotlib', 'seaborn', 'xgboost', 'shap', 'lime']
versions = {}

for lib in libraries:
    # Map import names to pip packaging name differences where appropriate
    import_name = 'sklearn' if lib == 'scikit-learn' else lib
    try:
        imported_lib = __import__(import_name)
        versions[lib] = getattr(imported_lib, '__version__', 'installed')
    except ImportError:
        versions[lib] = 'not installed'

# Save the information to environment_versions.txt
env_file_path = os.path.join(base_path, 'logs/environment_versions.txt')
with open(env_file_path, 'w') as f:
    f.write(f"Python version: {python_version}\n")
    for lib, ver in versions.items():
        f.write(f"{lib}: {ver}\n")

print(f"\nSaved environment metadata to: {env_file_path}")

Python Version: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]

Saved environment metadata to: /content/drive/MyDrive/FarmSight_XAI/logs/environment_versions.txt


In [ ]:
# Validation
all_folders_exist = all(os.path.exists(folder) for folder in folders_to_create)
file_exists = os.path.exists(env_file_path)

print("--- Validation Summary ---")
print(f"All directories exist: {all_folders_exist}")
print(f"Environment file exists: {file_exists}")

if all_folders_exist and file_exists:
    print("\nSETUP VERIFIED")
else:
    print("\nSETUP FAILED - Check output logs above.")

--- Validation Summary ---
All directories exist: True
Environment file exists: True

SETUP VERIFIED


## Data Acquisition & Verification Stage
This section manages downloading the World Bank agricultural dataset, extracting the designated ICRISAT apportioned crop data, verifying its properties, and documenting data provenance.

In [ ]:
import requests
import hashlib
import zipfile
import os

# Configure URL and paths
zip_url = "https://reproducibility.worldbank.org/catalog/409/download/1152/RR_IND_2025_488.zip"
temp_zip_path = "/content/RR_IND_2025_488.zip"

# 1. & 2. Download the ZIP package with streaming and verify HTTP status
print(f"Downloading {zip_url} ...")
response = requests.get(zip_url, stream=True)
if response.status_code != 200:
    raise RuntimeError(f"Download failed with HTTP status code: {response.status_code}")

# 3. Save the downloaded ZIP temporarily under /content/
with open(temp_zip_path, 'wb') as f:
    for chunk in response.iter_content(chunk_size=1024 * 1024):
        if chunk:
            f.write(chunk)

# 4. Print size in MB and calculate SHA-256 checksum
file_size_mb = os.path.getsize(temp_zip_path) / (1024 * 1024)
print(f"Downloaded ZIP Size: {file_size_mb:.2f} MB")

sha256_hash = hashlib.sha256()
with open(temp_zip_path, 'rb') as f:
    for byte_block in iter(lambda: f.read(4096), b""):
        sha256_hash.update(byte_block)
zip_sha256 = sha256_hash.hexdigest()
print(f"ZIP SHA-256 Checksum: {zip_sha256}")

Downloaded ZIP Size: 329.80 MB
ZIP SHA-256 Checksum: f9ff83879f0065a7563aedccdb4590fa224eb1c75519cc725c35a9ae5f5d18cd


In [ ]:
# 5. & 6. Inspect ZIP contents without fully extracting and search for the target file
target_suffix = "Crops_Area Production Yield.csv"
target_path_sub = "raw/DLD/Apportioned/"
target_zip_path = None

with zipfile.ZipFile(temp_zip_path, 'r') as zip_ref:
    for file_info in zip_ref.infolist():
        if file_info.filename.endswith(target_suffix) and target_path_sub in file_info.filename:
            target_zip_path = file_info.filename
            break

if not target_zip_path:
    raise FileNotFoundError("Specified target apportioned crop CSV file was not found inside the ZIP package.")

print(f"Found target CSV inside ZIP: {target_zip_path}")

# 7. & 8. Extract only that CSV and save to raw data directory
destination_csv_path = "/content/drive/MyDrive/FarmSight_XAI/data/raw/icrisat_dld_crops_area_production_yield_apportioned.csv"

with zipfile.ZipFile(temp_zip_path, 'r') as zip_ref:
    with zip_ref.open(target_zip_path) as source_file:
        with open(destination_csv_path, 'wb') as dest_file:
            dest_file.write(source_file.read())

print(f"Extracted and saved to: {destination_csv_path}")

Found target CSV inside ZIP: RR_IND_2025_488/Reproducbility Package/raw/DLD/Apportioned/Crops_Area Production Yield.csv
Extracted and saved to: /content/drive/MyDrive/FarmSight_XAI/data/raw/icrisat_dld_crops_area_production_yield_apportioned.csv


In [ ]:
import pandas as pd
from datetime import datetime, timezone

# 10. Load the saved CSV only for verification
df = pd.read_csv(destination_csv_path)

# Calculate CSV SHA-256
csv_hash = hashlib.sha256()
with open(destination_csv_path, 'rb') as f:
    for byte_block in iter(lambda: f.read(4096), b""):
        csv_hash.update(byte_block)
csv_sha256 = csv_hash.hexdigest()

# 11. Print required data insights
csv_size_mb = os.path.getsize(destination_csv_path) / (1024 * 1024)
row_count, col_count = df.shape

print(f"Saved Path: {destination_csv_path}")
print(f"File Size: {csv_size_mb:.2f} MB")
print(f"Row Count: {row_count}")
print(f"Column Count: {col_count}")
print("\nComplete Column-Name List:")
print(df.columns.tolist())
print("\nFirst 5 Rows:")
display(df.head())
print("\nMissing-Value Count for Each Column:")
print(df.isnull().sum())

Saved Path: /content/drive/MyDrive/FarmSight_XAI/data/raw/icrisat_dld_crops_area_production_yield_apportioned.csv
File Size: 8.83 MB
Row Count: 16146
Column Count: 80

Complete Column-Name List:
['Dist Code', 'Year', 'State Code', 'State Name', 'Dist Name', 'RICE AREA (1000 ha)', 'RICE PRODUCTION (1000 tons)', 'RICE YIELD (Kg per ha)', 'WHEAT AREA (1000 ha)', 'WHEAT PRODUCTION (1000 tons)', 'WHEAT YIELD (Kg per ha)', 'KHARIF SORGHUM AREA (1000 ha)', 'KHARIF SORGHUM PRODUCTION (1000 tons)', 'KHARIF SORGHUM YIELD (Kg per ha)', 'RABI SORGHUM AREA (1000 ha)', 'RABI SORGHUM PRODUCTION (1000 tons)', 'RABI SORGHUM YIELD (Kg per ha)', 'SORGHUM AREA (1000 ha)', 'SORGHUM PRODUCTION (1000 tons)', 'SORGHUM YIELD (Kg per ha)', 'PEARL MILLET AREA (1000 ha)', 'PEARL MILLET PRODUCTION (1000 tons)', 'PEARL MILLET YIELD (Kg per ha)', 'MAIZE AREA (1000 ha)', 'MAIZE PRODUCTION (1000 tons)', 'MAIZE YIELD (Kg per ha)', 'FINGER MILLET AREA (1000 ha)', 'FINGER MILLET PRODUCTION (1000 tons)', 'FINGER MILLET YI

,Dist Code,Year,State Code,State Name,Dist Name,RICE AREA (1000 ha),RICE PRODUCTION (1000 tons),RICE YIELD (Kg per ha),WHEAT AREA (1000 ha),WHEAT PRODUCTION (1000 tons),...,SUGARCANE YIELD (Kg per ha),COTTON AREA (1000 ha),COTTON PRODUCTION (1000 tons),COTTON YIELD (Kg per ha),FRUITS AREA (1000 ha),VEGETABLES AREA (1000 ha),FRUITS AND VEGETABLES AREA (1000 ha),POTATOES AREA (1000 ha),ONION AREA (1000 ha),FODDER AREA (1000 ha)
0,1,1966,14,Chhattisgarh,Durg,548.0,185.0,337.59,44.0,20.0,...,1777.78,0.0,0.0,0.0,5.95,6.64,12.59,0.01,0.60,0.47
1,1,1967,14,Chhattisgarh,Durg,547.0,409.0,747.71,50.0,26.0,...,1500.00,0.0,0.0,0.0,5.77,7.24,13.02,0.01,0.56,1.23
2,1,1968,14,Chhattisgarh,Durg,556.3,468.0,841.27,53.7,30.0,...,1000.00,0.0,0.0,0.0,5.41,7.40,12.81,0.10,0.58,1.02
3,1,1969,14,Chhattisgarh,Durg,563.4,400.8,711.40,49.4,26.5,...,1900.00,0.0,0.0,0.0,5.52,7.16,12.69,0.01,0.56,0.84
4,1,1970,14,Chhattisgarh,Durg,571.6,473.6,828.55,44.2,29.0,...,2000.00,0.0,0.0,0.0,5.45,7.19,12.64,0.02,0.52,0.42



Missing-Value Count for Each Column:
Dist Code                               0
Year                                    0
State Code                              0
State Name                              0
Dist Name                               0
                                       ..
VEGETABLES AREA (1000 ha)               0
FRUITS AND VEGETABLES AREA (1000 ha)    0
POTATOES AREA (1000 ha)                 0
ONION AREA (1000 ha)                    0
FODDER AREA (1000 ha)                   0
Length: 80, dtype: int64


In [ ]:
# 12. Create icrisat_data_provenance.txt
provenance_file_path = "/content/drive/MyDrive/FarmSight_XAI/logs/icrisat_data_provenance.txt"
current_utc_time = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")

provenance_content = f"""Original Package URL: {zip_url}
Catalog Page URL: https://reproducibility.worldbank.org/index.php/catalog/409
Original Internal ZIP Path: {target_zip_path}
Extraction Date & Time: {current_utc_time}
Extracted CSV Filename: icrisat_dld_crops_area_production_yield_apportioned.csv
Extracted CSV SHA-256 Checksum: {csv_sha256}
Row Count: {row_count}
Column Count: {col_count}
Stated Source: ICRISAT and Tata-Cornell Institute District Level Database
Access Route: World Bank reproducibility package
"""

with open(provenance_file_path, 'w') as f:
    f.write(provenance_content)

print(f"Provenance logged successfully to: {provenance_file_path}")

Provenance logged successfully to: /content/drive/MyDrive/FarmSight_XAI/logs/icrisat_data_provenance.txt


In [ ]:
# Validation checks
csv_exists = os.path.exists(destination_csv_path)
prov_exists = os.path.exists(provenance_file_path)
valid_dims = False

if csv_exists:
    # Read minimal layout to check structure
    test_df = pd.read_csv(destination_csv_path, nrows=1)
    valid_dims = test_df.shape[0] >= 0 and test_df.shape[1] >= 1 # At least one row and one column setup checked

print("\n--- Acquisition Validation Summary ---")
print(f"CSV File Exists: {csv_exists}")
print(f"CSV Contains Rows/Cols: {valid_dims}")
print(f"Provenance File Exists: {prov_exists}")

if csv_exists and valid_dims and prov_exists:
    print("\nICRISAT ACQUISITION VERIFIED")
else:
    print("\nACQUISITION VALIDATION FAILED")


--- Acquisition Validation Summary ---
CSV File Exists: True
CSV Contains Rows/Cols: True
Provenance File Exists: True

ICRISAT ACQUISITION VERIFIED


## Read-Only Dataset Audit Stage
This section performs a descriptive audit of the raw ICRISAT dataset, saves a structural inventory, and writes a metadata summary log without modifying the original data.

In [ ]:
import os
import pandas as pd
import numpy as np

raw_path = "/content/drive/MyDrive/FarmSight_XAI/data/raw/icrisat_dld_crops_area_production_yield_apportioned.csv"

# 1. Load the CSV without changing original file
df_audit = pd.read_csv(raw_path)

# 2. Print exact shape, column list, data types, memory, duplicates and missing metrics
print(f"Dataset Shape: {df_audit.shape}")
print(f"Memory Usage (Total): {df_audit.memory_usage(deep=True).sum() / (1024**2):.2f} MB")
print(f"Number of Exact Duplicate Rows: {df_audit.duplicated().sum()}")

print("\n--- Column Data Types & Missing-Value Statistics ---")
missing_counts = df_audit.isnull().sum()
missing_pct = (missing_counts / len(df_audit)) * 100

audit_cols_df = pd.DataFrame({
    'Dtype': df_audit.dtypes,
    'Missing_Count': missing_counts,
    'Missing_Percentage': missing_pct
})
pd.set_option('display.max_rows', 100)
display(audit_cols_df)

Dataset Shape: (16146, 80)
Memory Usage (Total): 11.41 MB
Number of Exact Duplicate Rows: 0

--- Column Data Types & Missing-Value Statistics ---


,Dtype,Missing_Count,Missing_Percentage
Dist Code,int64,0,0.0
Year,int64,0,0.0
State Code,int64,0,0.0
State Name,object,0,0.0
Dist Name,object,0,0.0
RICE AREA (1000 ha),float64,0,0.0
RICE PRODUCTION (1000 tons),float64,0,0.0
RICE YIELD (Kg per ha),float64,0,0.0
WHEAT AREA (1000 ha),float64,0,0.0
WHEAT PRODUCTION (1000 tons),float64,0,0.0


In [ ]:
# 3. Identify and print identifier columns
id_cols = {
    'district_code': 'Dist Code',
    'district_name': 'Dist Name',
    'state_code': 'State Code',
    'state_name': 'State Name',
    'year': 'Year'
}
print("Identifier Columns map:", id_cols)

# 4. Temporal and spatial span metrics
min_year = int(df_audit['Year'].min())
max_year = int(df_audit['Year'].max())
num_years = df_audit['Year'].nunique()
num_states = df_audit['State Name'].nunique()
num_districts = df_audit['Dist Name'].nunique() # Unique names
num_dist_codes = df_audit['Dist Code'].nunique() # Unique codes

print(f"Minimum Year: {min_year}")
print(f"Maximum Year: {max_year}")
print(f"Number of Unique Years: {num_years}")
print(f"Number of Unique States: {num_states}")
print(f"Number of Unique Districts (by Name): {num_districts}")
print(f"Number of Unique Districts (by Code): {num_dist_codes}")

Identifier Columns map: {'district_code': 'Dist Code', 'district_name': 'Dist Name', 'state_code': 'State Code', 'state_name': 'State Name', 'year': 'Year'}
Minimum Year: 1966
Maximum Year: 2017
Number of Unique Years: 52
Number of Unique States: 20
Number of Unique Districts (by Name): 311
Number of Unique Districts (by Code): 311


In [ ]:
# 5. List all columns containing AREA, PRODUCTION or YIELD in their names
metric_cols = [c for c in df_audit.columns if any(kw in c for kw in ['AREA', 'PRODUCTION', 'YIELD'])]
print(f"Found {len(metric_cols)} metric columns.")

# 6. For each crop-yield column, count: non-missing, zero, negative values
yield_cols = [c for c in df_audit.columns if 'YIELD' in c]
yield_stats = []

for col in yield_cols:
    non_missing = df_audit[col].notnull().sum()
    zeros = (df_audit[col] == 0).sum()
    negatives = (df_audit[col] < 0).sum()
    yield_stats.append({
        'Column': col,
        'Non_Missing': non_missing,
        'Zeros': zeros,
        'Negatives': negatives
    })

yield_stats_df = pd.DataFrame(yield_stats)
display(yield_stats_df)

Found 75 metric columns.


,Column,Non_Missing,Zeros,Negatives
0,RICE YIELD (Kg per ha),16146,1143,22
1,WHEAT YIELD (Kg per ha),16146,2369,35
2,KHARIF SORGHUM YIELD (Kg per ha),16146,4864,40
3,RABI SORGHUM YIELD (Kg per ha),16146,12113,40
4,SORGHUM YIELD (Kg per ha),16146,4541,40
5,PEARL MILLET YIELD (Kg per ha),16146,6133,54
6,MAIZE YIELD (Kg per ha),16146,1964,28
7,FINGER MILLET YIELD (Kg per ha),16146,10243,20
8,BARLEY YIELD (Kg per ha),16146,8198,20
9,CHICKPEA YIELD (Kg per ha),16146,2235,42


In [ ]:
# 7. Print the first 5 rows using identifier columns + Rice, Maize and Cotton area, prod, yield
target_keywords = ['RICE', 'MAIZE', 'COTTON']
selected_metrics = [c for c in df_audit.columns if any(k in c for k in target_keywords)]
print_cols = list(id_cols.values()) + selected_metrics

print("Displaying subset of target crops (Rice, Maize, Cotton):")
display(df_audit[print_cols].head(5))

Displaying subset of target crops (Rice, Maize, Cotton):


,Dist Code,Dist Name,State Code,State Name,Year,RICE AREA (1000 ha),RICE PRODUCTION (1000 tons),RICE YIELD (Kg per ha),MAIZE AREA (1000 ha),MAIZE PRODUCTION (1000 tons),MAIZE YIELD (Kg per ha),COTTON AREA (1000 ha),COTTON PRODUCTION (1000 tons),COTTON YIELD (Kg per ha)
0,1,Durg,14,Chhattisgarh,1966,548.0,185.0,337.59,3.0,2.0,666.67,0.0,0.0,0.0
1,1,Durg,14,Chhattisgarh,1967,547.0,409.0,747.71,3.0,3.0,1000.00,0.0,0.0,0.0
2,1,Durg,14,Chhattisgarh,1968,556.3,468.0,841.27,2.8,2.0,714.29,0.0,0.0,0.0
3,1,Durg,14,Chhattisgarh,1969,563.4,400.8,711.40,2.7,2.3,851.85,0.0,0.0,0.0
4,1,Durg,14,Chhattisgarh,1970,571.6,473.6,828.55,2.5,3.3,1320.00,0.0,0.0,0.0


In [ ]:
# 8. Save a complete column inventory to: results/tables/icrisat_column_inventory.csv
inventory_path = "/content/drive/MyDrive/FarmSight_XAI/results/tables/icrisat_column_inventory.csv"
audit_cols_df.reset_index().rename(columns={'index': 'Column_Name'}).to_csv(inventory_path, index=False)
print(f"Column inventory successfully written to: {inventory_path}")

Column inventory successfully written to: /content/drive/MyDrive/FarmSight_XAI/results/tables/icrisat_column_inventory.csv


In [ ]:
# 9. Save the audit summary to: logs/icrisat_raw_audit.txt
audit_log_path = "/content/drive/MyDrive/FarmSight_XAI/logs/icrisat_raw_audit.txt"

audit_summary_text = f"""FarmSight XAI ICRISAT Raw Data Audit
=========================================
Dataset Path: {raw_path}
Shape: {df_audit.shape[0]} rows, {df_audit.shape[1]} columns
Total Memory Usage: {df_audit.memory_usage(deep=True).sum() / (1024**2):.2f} MB
Duplicate Rows: {df_audit.duplicated().sum()}

Temporal & Spatial Span:
------------------------
Year Range: {min_year} - {max_year} ({num_years} unique years)
Unique States: {num_states}
Unique Districts (by Code): {num_dist_codes}

Metric Breakdown:
-----------------
Total Area/Production/Yield Columns: {len(metric_cols)}
Total Yield Columns Analyzed: {len(yield_cols)}
"""

with open(audit_log_path, 'w') as f:
    f.write(audit_summary_text)
    f.write("\nYield Columns Statistics:\n")
    f.write(yield_stats_df.to_string(index=False))

print(f"Audit summary written to: {audit_log_path}")

Audit summary written to: /content/drive/MyDrive/FarmSight_XAI/logs/icrisat_raw_audit.txt


In [ ]:
# Validation checks
import hashlib

# Confirm the raw CSV is structurally identical to the loaded context (not modified)
original_hash_expected = '7857decb1a196a54aac71a0f841af43c3bac98b3b90744aadca18b1859cea6f3'
current_raw_hash = hashlib.sha256()
with open(raw_path, 'rb') as f:
    for byte_block in iter(lambda: f.read(4096), b""):
        current_raw_hash.update(byte_block)
raw_unmodified = (current_raw_hash.hexdigest() == original_hash_expected)

# Confirm file generation
inventory_exists = os.path.exists(inventory_path) and os.path.getsize(inventory_path) > 0
audit_log_exists = os.path.exists(audit_log_path) and os.path.getsize(audit_log_path) > 0

print("\n--- Audit Verification Summary ---")
print(f"Raw CSV File Unmodified: {raw_unmodified}")
print(f"Inventory CSV Created: {inventory_exists}")
print(f"Audit Log Created: {audit_log_exists}")

if raw_unmodified and inventory_exists and audit_log_exists:
    print("\nICRISAT RAW AUDIT VERIFIED")
else:
    print("\nAUDIT VERIFICATION FAILED")


--- Audit Verification Summary ---
Raw CSV File Unmodified: True
Inventory CSV Created: True
Audit Log Created: True

ICRISAT RAW AUDIT VERIFIED


## Candidate Crop Coverage Analysis Stage
This section conducts a targeted read-only audit of spatial and temporal coverage for Rice, Maize, and Cotton to assist in scoping our experiments.

In [ ]:
import os
import pandas as pd

raw_path = "/content/drive/MyDrive/FarmSight_XAI/data/raw/icrisat_dld_crops_area_production_yield_apportioned.csv"
df_cov = pd.read_csv(raw_path)

candidates = ['RICE', 'MAIZE', 'COTTON']
crop_cols = {}

# 1. Identify the exact columns using case-insensitive match
for crop in candidates:
    crop_cols[crop] = {
        'area': [c for c in df_cov.columns if crop in c.upper() and 'AREA' in c.upper()][0],
        'production': [c for c in df_cov.columns if crop in c.upper() and 'PRODUCTION' in c.upper()][0],
        'yield': [c for c in df_cov.columns if crop in c.upper() and 'YIELD' in c.upper()][0]
    }

print("Matched Columns:")
for crop, cols in crop_cols.items():
    print(f"{crop}: {cols}")

Matched Columns:
RICE: {'area': 'RICE AREA (1000 ha)', 'production': 'RICE PRODUCTION (1000 tons)', 'yield': 'RICE YIELD (Kg per ha)'}
MAIZE: {'area': 'MAIZE AREA (1000 ha)', 'production': 'MAIZE PRODUCTION (1000 tons)', 'yield': 'MAIZE YIELD (Kg per ha)'}
COTTON: {'area': 'COTTON AREA (1000 ha)', 'production': 'COTTON PRODUCTION (1000 tons)', 'yield': 'COTTON YIELD (Kg per ha)'}


In [ ]:
# 2. & 3. Calculate coverage and era-wise breakdowns
coverage_stats = []
era_bins = [
    (1966, 1989, "1966-1989"),
    (1990, 1999, "1990-1999"),
    (2000, 2009, "2000-2009"),
    (2010, 2017, "2010-2017")
]

for crop in candidates:
    y_col = crop_cols[crop]['yield']

    total_rows = len(df_cov)
    non_missing_yield = df_cov[y_col].notnull().sum()
    positive_yield = (df_cov[y_col] > 0).sum()
    zero_yield = (df_cov[y_col] == 0).sum()
    negative_yield = (df_cov[y_col] < 0).sum()

    # Filtering for positive yield subset
    pos_df = df_cov[df_cov[y_col] > 0]
    earliest_year = int(pos_df['Year'].min()) if not pos_df.empty else None
    latest_year = int(pos_df['Year'].max()) if not pos_df.empty else None
    unique_states = pos_df['State Name'].nunique() if not pos_df.empty else 0
    unique_districts = pos_df['Dist Name'].nunique() if not pos_df.empty else 0

    # Era breakdown for positive yield records
    era_counts = {}
    for start, end, label in era_bins:
        era_counts[label] = ((pos_df['Year'] >= start) & (pos_df['Year'] <= end)).sum()

    stat_dict = {
        'Crop': crop.capitalize(),
        'Total_Rows': total_rows,
        'Non_Missing_Yield': non_missing_yield,
        'Positive_Yield': positive_yield,
        'Zero_Yield': zero_yield,
        'Negative_Yield': negative_yield,
        'Earliest_Pos_Year': earliest_year,
        'Latest_Pos_Year': latest_year,
        'Unique_States_Pos': unique_states,
        'Unique_Districts_Pos': unique_districts
    }
    # Merge era counts
    for label, count in era_counts.items():
        stat_dict[f'Pos_Yield_Rows_{label}'] = count

    coverage_stats.append(stat_dict)

coverage_df = pd.DataFrame(coverage_stats)
print("\n--- Candidate Crop Coverage Summary ---")
display(coverage_df)


--- Candidate Crop Coverage Summary ---


,Crop,Total_Rows,Non_Missing_Yield,Positive_Yield,Zero_Yield,Negative_Yield,Earliest_Pos_Year,Latest_Pos_Year,Unique_States_Pos,Unique_Districts_Pos,Pos_Yield_Rows_1966-1989,Pos_Yield_Rows_1990-1999,Pos_Yield_Rows_2000-2009,Pos_Yield_Rows_2010-2017
0,Rice,16146,16146,14981,1143,22,1966,2017,20,307,6952,2907,2874,2248
1,Maize,16146,16146,14154,1964,28,1966,2017,20,302,6382,2715,2797,2260
2,Cotton,16146,16146,7816,8315,15,1966,2017,19,231,3648,1525,1479,1164


In [ ]:
# 5. & 6. Save outputs to results and log directory
table_path = "/content/drive/MyDrive/FarmSight_XAI/results/tables/candidate_crop_coverage.csv"
report_path = "/content/drive/MyDrive/FarmSight_XAI/logs/candidate_crop_coverage.txt"

# Save csv table
coverage_df.to_csv(table_path, index=False)
print(f"Saved comparative table to: {table_path}")

# Generate and save text report
report_content = f"""FarmSight XAI Candidate Crop Coverage Report
==============================================
Raw Dataset evaluated: {raw_path}
Target Crops: Rice, Maize, Cotton

"""
report_content += coverage_df.to_string(index=False)

with open(report_path, 'w') as f:
    f.write(report_content)
print(f"Saved text report to: {report_path}")

Saved comparative table to: /content/drive/MyDrive/FarmSight_XAI/results/tables/candidate_crop_coverage.csv
Saved text report to: /content/drive/MyDrive/FarmSight_XAI/logs/candidate_crop_coverage.txt


In [ ]:
# Validation checks
import hashlib

original_hash_expected = '7857decb1a196a54aac71a0f841af43c3bac98b3b90744aadca18b1859cea6f3'
current_raw_hash = hashlib.sha256()
with open(raw_path, 'rb') as f:
    for byte_block in iter(lambda: f.read(4096), b""):
        current_raw_hash.update(byte_block)
raw_unmodified = (current_raw_hash.hexdigest() == original_hash_expected)

table_valid = os.path.exists(table_path) and os.path.getsize(table_path) > 0
report_valid = os.path.exists(report_path) and os.path.getsize(report_path) > 0

print("\n--- Verification Summary ---")
print(f"Raw dataset hash unchanged: {raw_unmodified}")
print(f"Coverage CSV table exists & non-empty: {table_valid}")
print(f"Coverage text report exists & non-empty: {report_valid}")

if raw_unmodified and table_valid and report_valid:
    print("\nCANDIDATE CROP COVERAGE VERIFIED")
else:
    print("\nCOVERAGE VERIFICATION FAILED")


--- Verification Summary ---
Raw dataset hash unchanged: True
Coverage CSV table exists & non-empty: True
Coverage text report exists & non-empty: True

CANDIDATE CROP COVERAGE VERIFIED


In [ ]:
import os
import hashlib
import pandas as pd
import numpy as np

raw_path = "/content/drive/MyDrive/FarmSight_XAI/data/raw/icrisat_dld_crops_area_production_yield_apportioned.csv"
interim_path = "/content/drive/MyDrive/FarmSight_XAI/data/interim/icrisat_rice_maize_cotton_long.csv"
conflicting_path = "/content/drive/MyDrive/FarmSight_XAI/results/tables/icrisat_conflicting_duplicates.csv"
report_path = "/content/drive/MyDrive/FarmSight_XAI/logs/icrisat_restructuring_report.txt"

# Expected raw SHA-256 to ensure read-only safety
expected_raw_hash = "7857decb1a196a54aac71a0f841af43c3bac98b3b90744aadca18b1859cea6f3"

def get_sha256(filepath):
    sha256 = hashlib.sha256()
    with open(filepath, 'rb') as f:
        for byte_block in iter(lambda: f.read(4096), b""):
            sha256.update(byte_block)
    return sha256.hexdigest()

# 1. Load the raw CSV
df_raw = pd.read_csv(raw_path)
raw_shape = df_raw.shape
print(f"Raw dataset loaded. Shape: {raw_shape}")

Raw dataset loaded. Shape: (16146, 80)


In [ ]:
# 2. & 3. Column identifiers and crop map
id_cols = {
    'Dist Code': 'district_code',
    'Dist Name': 'district_name',
    'State Code': 'state_code',
    'State Name': 'state_name',
    'Year': 'year'
}

crop_cols_map = {
    'Rice': {
        'area': 'RICE AREA (1000 ha)',
        'production': 'RICE PRODUCTION (1000 tons)',
        'yield': 'RICE YIELD (Kg per ha)'
    },
    'Maize': {
        'area': 'MAIZE AREA (1000 ha)',
        'production': 'MAIZE PRODUCTION (1000 tons)',
        'yield': 'MAIZE YIELD (Kg per ha)'
    },
    'Cotton': {
        'area': 'COTTON AREA (1000 ha)',
        'production': 'COTTON PRODUCTION (1000 tons)',
        'yield': 'COTTON YIELD (Kg per ha)'
    }
}

# 4. & 5. Reshape wide to long
long_records = []
for index, row in df_raw.iterrows():
    base_info = {id_cols[k]: row[k] for k in id_cols}
    for crop, metrics in crop_cols_map.items():
        rec = base_info.copy()
        rec['crop'] = crop
        rec['area_1000_ha'] = row[metrics['area']]
        rec['production_1000_tons'] = row[metrics['production']]
        rec['yield_kg_per_ha'] = row[metrics['yield']]
        long_records.append(rec)

df_long_all = pd.DataFrame(long_records)

In [ ]:
# 6. Filter & count removals (missing, zero, negative yield)
removal_stats = []
df_cleaned_list = []

for crop in ['Rice', 'Maize', 'Cotton']:
    crop_df = df_long_all[df_long_all['crop'] == crop].copy()

    total_before = len(crop_df)
    missing_mask = crop_df['yield_kg_per_ha'].isnull()
    zero_mask = crop_df['yield_kg_per_ha'] == 0
    negative_mask = crop_df['yield_kg_per_ha'] < 0

    missing_count = missing_mask.sum()
    zero_count = zero_mask.sum()
    negative_count = negative_mask.sum()

    # Keep positive non-missing
    keep_mask = crop_df['yield_kg_per_ha'] > 0
    cleaned_crop_df = crop_df[keep_mask].copy()

    df_cleaned_list.append(cleaned_crop_df)

    removal_stats.append({
        'Crop': crop,
        'Initial': total_before,
        'Removed_Missing': missing_count,
        'Removed_Zero': zero_count,
        'Removed_Negative': negative_count,
        'Retained': len(cleaned_crop_df)
    })

df_cleaned = pd.concat(df_cleaned_list, ignore_index=True)
removals_df = pd.DataFrame(removal_stats)
print("Removal Breakdown per Crop:")
display(removals_df)

Removal Breakdown per Crop:


,Crop,Initial,Removed_Missing,Removed_Zero,Removed_Negative,Retained
0,Rice,16146,0,1143,22,14981
1,Maize,16146,0,1964,28,14154
2,Cotton,16146,0,8315,15,7816


In [ ]:
# 7. & 8. Check exact duplicate rows and duplicate keys
exact_dups = df_cleaned.duplicated().sum()
print(f"Number of exact duplicate rows: {exact_dups}")

# Remove identical duplicate rows
df_cleaned = df_cleaned.drop_duplicates().copy()

# Check for duplicate [district_code, year, crop] keys (conflict analysis)
key_cols = ['district_code', 'year', 'crop']
all_key_duplicates = df_cleaned[df_cleaned.duplicated(subset=key_cols, keep=False)]

# Separate completely conflicting keys
conflicting_records = pd.DataFrame(columns=df_cleaned.columns)
if not all_key_duplicates.empty:
    # Find rows where values differ but keys are identical
    grouped = all_key_duplicates.groupby(key_cols)
    conflicting_indices = []
    for keys, group in grouped:
        # If any of the variables differ within the grouped keys
        if group[['area_1000_ha', 'production_1000_tons', 'yield_kg_per_ha']].nunique().max() > 1:
            conflicting_indices.extend(group.index.tolist())

    if conflicting_indices:
        conflicting_records = df_cleaned.loc[conflicting_indices].copy()
        # Remove conflicting duplicates from our clean interim dataset
        df_cleaned = df_cleaned.drop(index=conflicting_indices).copy()

conflicting_count = len(conflicting_records)
print(f"Number of conflicting-key records isolated: {conflicting_count}")

Number of exact duplicate rows: 0
Number of conflicting-key records isolated: 0


In [ ]:
# 9, 10, 11 & 12. Yield Consistency calculation
# calculated_yield = production_1000_tons / area_1000_ha * 1000
# Checked only where area and production are positive (> 0)

calc_mask = (df_cleaned['area_1000_ha'] > 0) & (df_cleaned['production_1000_tons'] > 0)

df_cleaned['yield_consistency_flag'] = True
df_cleaned['percentage_difference'] = 0.0

if calc_mask.any():
    calculated_yield = (df_cleaned.loc[calc_mask, 'production_1000_tons'] / df_cleaned.loc[calc_mask, 'area_1000_ha']) * 1000
    reported_yield = df_cleaned.loc[calc_mask, 'yield_kg_per_ha']

    # Calculate percentage difference relative to reported yield (prevent divide by zero)
    pct_diff = np.where(
        reported_yield != 0,
        np.abs(reported_yield - calculated_yield) / reported_yield * 100,
        0.0
    )

    df_cleaned.loc[calc_mask, 'percentage_difference'] = pct_diff
    # Set flag to False if difference exceeds 10% tolerance limit
    df_cleaned.loc[calc_mask, 'yield_consistency_flag'] = pct_diff <= 10.0

flag_counts = df_cleaned['yield_consistency_flag'].value_counts()
print("Yield Consistency Flag Counts (True = within 10% tolerance, False = inconsistent):")
print(flag_counts)

Yield Consistency Flag Counts (True = within 10% tolerance, False = inconsistent):
yield_consistency_flag
True     36844
False      107
Name: count, dtype: int64


In [ ]:
# 13, 14, 15. Save outputs
os.makedirs(os.path.dirname(interim_path), exist_ok=True)
os.makedirs(os.path.dirname(conflicting_path), exist_ok=True)
os.makedirs(os.path.dirname(report_path), exist_ok=True)

df_cleaned.to_csv(interim_path, index=False)
conflicting_records.to_csv(conflicting_path, index=False)

# Generate restructuring report summary text
report_txt = f"""FarmSight XAI Multicrop Dataset Restructuring Report
=======================================================
Source Dataset: {raw_path}
Target Crops: Rice, Maize, Cotton

Quality Control Summary:
-----------------------
Raw File Dimensions: {raw_shape[0]} rows, {raw_shape[1]} columns
Final Cleaned Interim Table Dimensions: {df_cleaned.shape[0]} rows, {df_cleaned.shape[1]} columns
Exact Identical Rows Removed: {exact_dups}
Conflicting Duplicate Key Records Removed: {conflicting_count}

Removal Breakdown by Reason & Crop:
-----------------------------------
{removals_df.to_string(index=False)}

Yield Consistency Flag Counts (Tolerance 10%):
---------------------------------------------
{flag_counts.to_string()}
"""

with open(report_path, 'w') as f:
    f.write(report_txt)

print(f"Saved long interim table to: {interim_path}")
print(f"Saved conflicting duplicate keys table to: {conflicting_path}")
print(f"Saved detailed restructuring log to: {report_path}")

Saved long interim table to: /content/drive/MyDrive/FarmSight_XAI/data/interim/icrisat_rice_maize_cotton_long.csv
Saved conflicting duplicate keys table to: /content/drive/MyDrive/FarmSight_XAI/results/tables/icrisat_conflicting_duplicates.csv
Saved detailed restructuring log to: /content/drive/MyDrive/FarmSight_XAI/logs/icrisat_restructuring_report.txt


In [ ]:
# Print requested diagnostic details
print("\n--- Interim Restructuring Metrics ---")
print(f"Raw dataset shape: {raw_shape}")
print(f"Final interim dataset shape: {df_cleaned.shape}")
print(f"Exact Duplicate count: {exact_dups}")
print(f"Conflicting duplicate key count: {conflicting_count}")
print(f"Yield consistency flag distribution:\n{df_cleaned['yield_consistency_flag'].value_counts()}")

print("\nRows and Year Range per Crop in final interim table:")
for crop in ['Rice', 'Maize', 'Cotton']:
    subset = df_cleaned[df_cleaned['crop'] == crop]
    print(f"- {crop}: {len(subset)} rows, Year range: {subset['year'].min()} to {subset['year'].max()}")


--- Interim Restructuring Metrics ---
Raw dataset shape: (16146, 80)
Final interim dataset shape: (36951, 11)
Exact Duplicate count: 0
Conflicting duplicate key count: 0
Yield consistency flag distribution:
yield_consistency_flag
True     36844
False      107
Name: count, dtype: int64

Rows and Year Range per Crop in final interim table:
- Rice: 14981 rows, Year range: 1966 to 2017
- Maize: 14154 rows, Year range: 1966 to 2017
- Cotton: 7816 rows, Year range: 1966 to 2017


In [ ]:
# --- Verification Suite ---
raw_hash_now = get_sha256(raw_path)
hash_valid = (raw_hash_now == expected_raw_hash)
files_exist = os.path.exists(interim_path) and os.path.exists(conflicting_path) and os.path.exists(report_path)

crops_in_table = df_cleaned['crop'].unique().tolist()
only_selected_crops = set(crops_in_table).issubset({'Rice', 'Maize', 'Cotton'})

yield_positive = (df_cleaned['yield_kg_per_ha'] > 0).all()
yield_not_null = df_cleaned['yield_kg_per_ha'].notnull().all()

print("\n--- Verification Status ---")
print(f"Raw file hash unchanged: {hash_valid}")
print(f"All three output files exist: {files_exist}")
print(f"Contains only Rice, Maize, Cotton: {only_selected_crops} ({crops_in_table})")
print(f"All yield values are positive: {yield_positive}")
print(f"All yield values are non-null: {yield_not_null}")

if hash_valid and files_exist and only_selected_crops and yield_positive and yield_not_null:
    print("\nMULTICROP INTERIM TABLE VERIFIED")
else:
    print("\nVERIFICATION FAILED")


--- Verification Status ---
Raw file hash unchanged: True
All three output files exist: True
Contains only Rice, Maize, Cotton: True (['Rice', 'Maize', 'Cotton'])
All yield values are positive: True
All yield values are non-null: True

MULTICROP INTERIM TABLE VERIFIED


In [ ]:
import os
import re
import time
import hashlib
import requests
import pandas as pd
from difflib import SequenceMatcher

input_path = "/content/drive/MyDrive/FarmSight_XAI/data/interim/icrisat_rice_maize_cotton_long.csv"
accepted_path = "/content/drive/MyDrive/FarmSight_XAI/data/interim/district_coordinates_accepted.csv"
review_path = "/content/drive/MyDrive/FarmSight_XAI/results/tables/district_geocoding_review.csv"
log_path = "/content/drive/MyDrive/FarmSight_XAI/logs/district_geocoding_report.txt"

def file_sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as file:
        for block in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def normalize(value):
    value = str(value).lower().strip()
    value = re.sub(r"\bdistrict\b", "", value)
    value = re.sub(r"[^a-z0-9]+", " ", value)
    return " ".join(value.split())

state_aliases = {
    "orissa": "odisha",
    "uttaranchal": "uttarakhand",
    "pondicherry": "puducherry",
}

def similarity(a, b):
    return SequenceMatcher(None, normalize(a), normalize(b)).ratio()

raw_hash_before = file_sha256(input_path)
data = pd.read_csv(input_path)

districts = (
    data[
        ["district_code", "district_name", "state_code", "state_name"]
    ]
    .drop_duplicates()
    .sort_values(["state_name", "district_name"])
    .reset_index(drop=True)
)

records = []
endpoint = "https://geocoding-api.open-meteo.com/v1/search"

for index, row in districts.iterrows():
    district = str(row["district_name"]).strip()
    state_original = str(row["state_name"]).strip()
    state_query = state_aliases.get(state_original.lower(), state_original)

    queries = [f"{district}, {state_query}", district]
    candidates = []
    error_message = ""

    for query in queries:
        try:
            response = requests.get(
                endpoint,
                params={
                    "name": query,
                    "count": 10,
                    "language": "en",
                    "countryCode": "IN",
                    "format": "json",
                },
                timeout=30,
            )
            response.raise_for_status()
            candidates = response.json().get("results", [])
            if candidates:
                break
        except Exception as error:
            error_message = str(error)

        time.sleep(0.15)

    scored = []
    for candidate in candidates:
        location_fields = [
            candidate.get("name", ""),
            candidate.get("admin2", ""),
            candidate.get("admin3", ""),
            candidate.get("admin4", ""),
        ]

        district_score = max(
            similarity(district, value)
            for value in location_fields
            if value
        )

        state_score = similarity(
            state_query,
            candidate.get("admin1", "")
        )

        combined_score = (0.80 * district_score) + (0.20 * state_score)
        scored.append((combined_score, district_score, state_score, candidate))

    if scored:
        scored.sort(key=lambda item: item[0], reverse=True)
        combined_score, district_score, state_score, best = scored[0]

        accepted = (
            district_score >= 0.85
            and state_score >= 0.60
            and best.get("latitude") is not None
            and best.get("longitude") is not None
        )

        record = {
            **row.to_dict(),
            "matched_name": best.get("name"),
            "matched_admin1": best.get("admin1"),
            "matched_admin2": best.get("admin2"),
            "matched_admin3": best.get("admin3"),
            "latitude": best.get("latitude"),
            "longitude": best.get("longitude"),
            "timezone": best.get("timezone"),
            "feature_code": best.get("feature_code"),
            "district_score": round(district_score, 4),
            "state_score": round(state_score, 4),
            "combined_score": round(combined_score, 4),
            "candidate_count": len(candidates),
            "match_status": "accepted" if accepted else "needs_review",
            "error": error_message,
        }
    else:
        record = {
            **row.to_dict(),
            "matched_name": None,
            "matched_admin1": None,
            "matched_admin2": None,
            "matched_admin3": None,
            "latitude": None,
            "longitude": None,
            "timezone": None,
            "feature_code": None,
            "district_score": 0,
            "state_score": 0,
            "combined_score": 0,
            "candidate_count": 0,
            "match_status": "unresolved",
            "error": error_message,
        }

    records.append(record)

    if (index + 1) % 25 == 0 or index + 1 == len(districts):
        print(f"Processed {index + 1}/{len(districts)} districts")

    time.sleep(0.15)

geocoded = pd.DataFrame(records)
accepted = geocoded[geocoded["match_status"] == "accepted"].copy()
review = geocoded[geocoded["match_status"] != "accepted"].copy()

accepted.to_csv(accepted_path, index=False)
review.to_csv(review_path, index=False)

raw_hash_after = file_sha256(input_path)
raw_unchanged = raw_hash_before == raw_hash_after

summary = f"""
District records attempted: {len(districts)}
Accepted automatically: {len(accepted)}
Needs review or unresolved: {len(review)}
Raw interim file unchanged: {raw_unchanged}
Accepted file: {accepted_path}
Review file: {review_path}

Coordinate meaning:
Open-Meteo geocoded representative locations are used.
They are not claimed to be official district centroids.
"""

with open(log_path, "w", encoding="utf-8") as file:
    file.write(summary.strip() + "\n")

outputs_exist = all(
    os.path.exists(path) and os.path.getsize(path) > 0
    for path in [accepted_path, review_path, log_path]
)

print(summary)
print("Output files created:", outputs_exist)

if raw_unchanged and outputs_exist and len(accepted) + len(review) == len(districts):
    print("DISTRICT GEOCODING STAGE VERIFIED")
else:
    print("DISTRICT GEOCODING STAGE FAILED")

Processed 25/311 districts
Processed 50/311 districts
Processed 75/311 districts
Processed 100/311 districts
Processed 125/311 districts
Processed 150/311 districts
Processed 175/311 districts
Processed 200/311 districts
Processed 225/311 districts
Processed 250/311 districts
Processed 275/311 districts
Processed 300/311 districts
Processed 311/311 districts

District records attempted: 311
Accepted automatically: 229
Needs review or unresolved: 82
Raw interim file unchanged: True
Accepted file: /content/drive/MyDrive/FarmSight_XAI/data/interim/district_coordinates_accepted.csv
Review file: /content/drive/MyDrive/FarmSight_XAI/results/tables/district_geocoding_review.csv

Coordinate meaning:
Open-Meteo geocoded representative locations are used.
They are not claimed to be official district centroids.

Output files created: True
DISTRICT GEOCODING STAGE VERIFIED


In [ ]:
import pandas as pd

long_path = "/content/drive/MyDrive/FarmSight_XAI/data/interim/icrisat_rice_maize_cotton_long.csv"
coords_path = "/content/drive/MyDrive/FarmSight_XAI/data/interim/district_coordinates_accepted.csv"

agri_df = pd.read_csv(long_path)
coords_df = pd.read_csv(coords_path)

print("Agricultural table shape:", agri_df.shape)
print("Agricultural columns:", agri_df.columns.tolist())

print("\nAccepted coordinates shape:", coords_df.shape)
print("Coordinate columns:", coords_df.columns.tolist())

Agricultural table shape: (36951, 11)
Agricultural columns: ['district_code', 'district_name', 'state_code', 'state_name', 'year', 'crop', 'area_1000_ha', 'production_1000_tons', 'yield_kg_per_ha', 'yield_consistency_flag', 'percentage_difference']

Accepted coordinates shape: (229, 18)
Coordinate columns: ['district_code', 'district_name', 'state_code', 'state_name', 'matched_name', 'matched_admin1', 'matched_admin2', 'matched_admin3', 'latitude', 'longitude', 'timezone', 'feature_code', 'district_score', 'state_score', 'combined_score', 'candidate_count', 'match_status', 'error']


In [ ]:
from pathlib import Path
import hashlib
import pandas as pd

base = Path("/content/drive/MyDrive/FarmSight_XAI")
long_path = base / "data/interim/icrisat_rice_maize_cotton_long.csv"
coords_path = base / "data/interim/district_coordinates_accepted.csv"

crop_output = base / "results/tables/geocoding_coverage_by_crop.csv"
state_output = base / "results/tables/geocoding_coverage_by_state.csv"
log_output = base / "logs/geocoding_coverage_report.txt"

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as file:
        for block in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

hash_before = sha256_file(long_path)

agri = pd.read_csv(long_path)
coords = pd.read_csv(coords_path)

join_keys = ["state_code", "district_code"]

# Standardize ID types without modifying saved source files.
for column in join_keys:
    agri[column] = pd.to_numeric(agri[column], errors="raise").astype("Int64")
    coords[column] = pd.to_numeric(coords[column], errors="raise").astype("Int64")

duplicate_coordinate_keys = int(coords.duplicated(join_keys).sum())

coord_keys = coords[join_keys].drop_duplicates().copy()
coord_keys["coordinate_available"] = True

merged = agri.merge(
    coord_keys,
    on=join_keys,
    how="left",
    validate="many_to_one"
)

merged["coordinate_available"] = (
    merged["coordinate_available"].fillna(False).astype(bool)
)

def coverage_summary(group):
    total_rows = len(group)
    covered_rows = int(group["coordinate_available"].sum())

    total_districts = len(group[join_keys].drop_duplicates())
    covered_districts = len(
        group.loc[group["coordinate_available"], join_keys].drop_duplicates()
    )

    return {
        "total_rows": total_rows,
        "covered_rows": covered_rows,
        "row_coverage_pct": round(100 * covered_rows / total_rows, 2),
        "total_districts": total_districts,
        "covered_districts": covered_districts,
        "district_coverage_pct": round(
            100 * covered_districts / total_districts, 2
        ),
    }

crop_rows = []
for crop, group in merged.groupby("crop", dropna=False):
    crop_rows.append({"crop": crop, **coverage_summary(group)})

crop_coverage = pd.DataFrame(crop_rows).sort_values("crop")

state_rows = []
for (state_code, state_name), group in merged.groupby(
    ["state_code", "state_name"], dropna=False
):
    state_rows.append({
        "state_code": state_code,
        "state_name": state_name,
        **coverage_summary(group)
    })

state_coverage = (
    pd.DataFrame(state_rows)
    .sort_values(["district_coverage_pct", "state_name"])
)

overall = coverage_summary(merged)
states_below_70 = state_coverage[
    state_coverage["district_coverage_pct"] < 70
]

crop_output.parent.mkdir(parents=True, exist_ok=True)
log_output.parent.mkdir(parents=True, exist_ok=True)

crop_coverage.to_csv(crop_output, index=False)
state_coverage.to_csv(state_output, index=False)

hash_after = sha256_file(long_path)

report = f"""FarmSight XAI Geocoding Coverage Audit
Agricultural rows: {len(agri)}
Accepted coordinate rows: {len(coords)}
Duplicate coordinate keys: {duplicate_coordinate_keys}

Overall covered agricultural rows: {overall['covered_rows']}
Overall row coverage (%): {overall['row_coverage_pct']}
Overall covered districts: {overall['covered_districts']}
Overall districts: {overall['total_districts']}
Overall district coverage (%): {overall['district_coverage_pct']}

States below 70% district coverage: {len(states_below_70)}
Agricultural file hash unchanged: {hash_before == hash_after}

Coordinate interpretation:
Open-Meteo geocoded representative locations are used.
They are not claimed to be official district centroids.
"""

log_output.write_text(report, encoding="utf-8")

files_exist = all(
    path.exists() for path in [crop_output, state_output, log_output]
)

verified = (
    len(merged) == len(agri)
    and duplicate_coordinate_keys == 0
    and overall["covered_rows"] > 0
    and hash_before == hash_after
    and files_exist
)

print("Overall coverage:")
print(pd.Series(overall).to_string())

print("\nCoverage by crop:")
display(crop_coverage)

print("\nStates below 70% district coverage:")
display(states_below_70)

print("\nOutput files created:", files_exist)
print("Agricultural file hash unchanged:", hash_before == hash_after)
print("Duplicate coordinate keys:", duplicate_coordinate_keys)

if verified:
    print("\nGEOCODING COVERAGE AUDIT VERIFIED")
else:
    print("\nGEOCODING COVERAGE AUDIT FAILED")

Overall coverage:
total_rows               36951.00
covered_rows             27026.00
row_coverage_pct            73.14
total_districts            311.00
covered_districts          229.00
district_coverage_pct       73.63

Coverage by crop:


/tmp/ipykernel_607/945274795.py:45: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  merged["coordinate_available"].fillna(False).astype(bool)


,crop,total_rows,covered_rows,row_coverage_pct,total_districts,covered_districts,district_coverage_pct
0,Cotton,7816,5459,69.84,231,167,72.29
1,Maize,14154,10567,74.66,302,221,73.18
2,Rice,14981,11000,73.43,307,227,73.94



States below 70% district coverage:


,state_code,state_name,total_rows,covered_rows,row_coverage_pct,total_districts,covered_districts,district_coverage_pct
10,11,Tamil Nadu,1558,623,39.99,12,4,33.33
0,1,Andhra Pradesh,1571,679,43.22,11,5,45.45
4,5,Karnataka,2619,1303,49.75,19,9,47.37
16,17,Assam,1301,595,45.73,10,5,50.00
14,15,Jharkhand,597,298,49.92,6,3,50.00
15,16,Uttarakhand,676,364,53.85,8,4,50.00
7,8,Orissa,1741,941,54.05,13,7,53.85
2,3,Gujarat,2335,1571,67.28,18,12,66.67
12,13,West Bengal,1498,975,65.09,15,10,66.67



Output files created: True
Agricultural file hash unchanged: True
Duplicate coordinate keys: 0

GEOCODING COVERAGE AUDIT VERIFIED


In [ ]:
from pathlib import Path
import calendar
import json
import requests
import pandas as pd

base = Path("/content/drive/MyDrive/FarmSight_XAI")

coords_path = base / "data/interim/district_coordinates_accepted.csv"
agri_path = base / "data/interim/icrisat_rice_maize_cotton_long.csv"

coords = pd.read_csv(coords_path)
agri = pd.read_csv(agri_path)

# Detect coordinate columns from the verified geocoding file.
lat_col = next(
    column for column in coords.columns
    if "latitude" in column.lower() or column.lower() == "lat"
)
lon_col = next(
    column for column in coords.columns
    if "longitude" in column.lower() or column.lower() in {"lon", "lng"}
)

sample = coords.iloc[0]

district_rows = agri[
    (agri["state_code"] == sample["state_code"]) &
    (agri["district_code"] == sample["district_code"])
]

sample_year = (
    2001 if 2001 in district_rows["year"].values
    else int(district_rows["year"].iloc[0])
)

params = {
    "latitude": float(sample[lat_col]),
    "longitude": float(sample[lon_col]),
    "start_date": f"{sample_year}-01-01",
    "end_date": f"{sample_year}-12-31",
    "daily": [
        "temperature_2m_mean",
        "temperature_2m_max",
        "temperature_2m_min",
        "precipitation_sum",
        "et0_fao_evapotranspiration"
    ],
    "timezone": "Asia/Kolkata",
    "models": "era5"
}

url = "https://archive-api.open-meteo.com/v1/archive"

response = requests.get(url, params=params, timeout=120)
response.raise_for_status()
payload = response.json()

daily = pd.DataFrame(payload["daily"])

expected_days = 366 if calendar.isleap(sample_year) else 365
missing_values = daily.isna().sum().to_dict()

pilot_csv = base / "results/tables/weather_pilot_daily.csv"
pilot_json = base / "data/raw/weather_pilot_response.json"
pilot_log = base / "logs/weather_pilot_report.txt"

daily.to_csv(pilot_csv, index=False)
pilot_json.write_text(json.dumps(payload, indent=2), encoding="utf-8")

verified = (
    len(daily) == expected_days
    and daily["time"].is_unique
    and pilot_csv.exists()
    and pilot_json.exists()
)

report = f"""Historical Weather Pilot
State: {sample['state_name']}
District: {sample['district_name']}
Year: {sample_year}
Model: ERA5
Latitude column: {lat_col}
Longitude column: {lon_col}
Expected days: {expected_days}
Received days: {len(daily)}
Missing values: {missing_values}
Pilot verified: {verified}
"""

pilot_log.write_text(report, encoding="utf-8")

print(report)
display(daily.head())

if verified:
    print("HISTORICAL WEATHER PILOT VERIFIED")
else:
    print("HISTORICAL WEATHER PILOT FAILED")

Historical Weather Pilot
State: Andhra Pradesh
District: Chittoor
Year: 2001
Model: ERA5
Latitude column: latitude
Longitude column: longitude
Expected days: 365
Received days: 365
Missing values: {'time': 0, 'temperature_2m_mean': 0, 'temperature_2m_max': 0, 'temperature_2m_min': 0, 'precipitation_sum': 0, 'et0_fao_evapotranspiration': 0}
Pilot verified: True



,time,temperature_2m_mean,temperature_2m_max,temperature_2m_min,precipitation_sum,et0_fao_evapotranspiration
0,2001-01-01,24.4,28.3,22.0,0.3,2.61
1,2001-01-02,23.9,26.8,21.6,0.5,2.35
2,2001-01-03,23.5,27.6,20.2,0.0,3.36
3,2001-01-04,22.9,28.1,17.8,0.0,3.07
4,2001-01-05,23.7,28.0,20.0,0.0,3.20


HISTORICAL WEATHER PILOT VERIFIED


In [ ]:
from pathlib import Path
import gzip
import json
import time
import requests
import pandas as pd

base = Path("/content/drive/MyDrive/FarmSight_XAI")

coords = pd.read_csv(
    base / "data/interim/district_coordinates_accepted.csv"
)
agri = pd.read_csv(
    base / "data/interim/icrisat_rice_maize_cotton_long.csv"
)

lat_col = next(
    c for c in coords.columns
    if "latitude" in c.lower() or c.lower() == "lat"
)
lon_col = next(
    c for c in coords.columns
    if "longitude" in c.lower() or c.lower() in {"lon", "lng"}
)

sample = coords.iloc[0]
start_year = int(agri["year"].min())
end_year = int(agri["year"].max())

params = {
    "latitude": float(sample[lat_col]),
    "longitude": float(sample[lon_col]),
    "start_date": f"{start_year}-01-01",
    "end_date": f"{end_year}-12-31",
    "daily": [
        "temperature_2m_mean",
        "temperature_2m_max",
        "temperature_2m_min",
        "precipitation_sum",
        "et0_fao_evapotranspiration"
    ],
    "timezone": "Asia/Kolkata",
    "models": "era5"
}

start_time = time.time()

response = requests.get(
    "https://archive-api.open-meteo.com/v1/archive",
    params=params,
    timeout=300
)
response.raise_for_status()
payload = response.json()

elapsed_seconds = round(time.time() - start_time, 2)

daily = pd.DataFrame(payload["daily"])
daily["date"] = pd.to_datetime(daily["time"])
daily["year"] = daily["date"].dt.year
daily["month"] = daily["date"].dt.month

daily["hot_day"] = daily["temperature_2m_max"] >= 35
daily["dry_day"] = daily["precipitation_sum"] < 1
daily["heavy_rain_day"] = daily["precipitation_sum"] >= 50

annual = daily.groupby("year").agg(
    annual_mean_temp_c=("temperature_2m_mean", "mean"),
    annual_max_temp_c=("temperature_2m_max", "max"),
    annual_min_temp_c=("temperature_2m_min", "min"),
    annual_precipitation_mm=("precipitation_sum", "sum"),
    annual_et0_mm=("et0_fao_evapotranspiration", "sum"),
    hot_days=("hot_day", "sum"),
    dry_days=("dry_day", "sum"),
    heavy_rain_days=("heavy_rain_day", "sum")
).reset_index()

monsoon = (
    daily[daily["month"].between(6, 9)]
    .groupby("year")
    .agg(
        monsoon_mean_temp_c=("temperature_2m_mean", "mean"),
        monsoon_precipitation_mm=("precipitation_sum", "sum"),
        monsoon_et0_mm=("et0_fao_evapotranspiration", "sum")
    )
    .reset_index()
)

annual = annual.merge(monsoon, on="year", how="left")

annual["annual_water_balance_mm"] = (
    annual["annual_precipitation_mm"] - annual["annual_et0_mm"]
)
annual["monsoon_water_balance_mm"] = (
    annual["monsoon_precipitation_mm"] - annual["monsoon_et0_mm"]
)

annual.insert(0, "district_name", sample["district_name"])
annual.insert(0, "district_code", sample["district_code"])
annual.insert(0, "state_name", sample["state_name"])
annual.insert(0, "state_code", sample["state_code"])
annual["latitude"] = float(sample[lat_col])
annual["longitude"] = float(sample[lon_col])

raw_output = base / "data/raw/weather_fullspan_pilot.json.gz"
annual_output = base / "results/tables/weather_fullspan_pilot_annual.csv"
log_output = base / "logs/weather_fullspan_pilot_report.txt"

with gzip.open(raw_output, "wt", encoding="utf-8") as file:
    json.dump(payload, file)

annual.to_csv(annual_output, index=False)

expected_years = end_year - start_year + 1
feature_columns = [
    column for column in annual.columns
    if column not in [
        "state_code", "state_name", "district_code",
        "district_name", "year", "latitude", "longitude"
    ]
]

verified = (
    len(annual) == expected_years
    and annual["year"].is_unique
    and annual[feature_columns].notna().all().all()
    and raw_output.exists()
    and annual_output.exists()
)

report = f"""Full-Span Historical Weather Pilot
District: {sample['district_name']}
State: {sample['state_name']}
Period: {start_year}-{end_year}
Daily rows received: {len(daily)}
Annual rows created: {len(annual)}
Expected annual rows: {expected_years}
Request time seconds: {elapsed_seconds}
Compressed raw bytes: {raw_output.stat().st_size}
Missing feature values: {int(annual[feature_columns].isna().sum().sum())}
Pilot verified: {verified}
"""

log_output.write_text(report, encoding="utf-8")

print(report)
display(annual.head())

if verified:
    print("FULL-SPAN WEATHER PILOT VERIFIED")
else:
    print("FULL-SPAN WEATHER PILOT FAILED")

Full-Span Historical Weather Pilot
District: Chittoor
State: Andhra Pradesh
Period: 1966-2017
Daily rows received: 18993
Annual rows created: 52
Expected annual rows: 52
Request time seconds: 6.02
Compressed raw bytes: 155924
Missing feature values: 0
Pilot verified: True



,state_code,state_name,district_code,district_name,year,annual_mean_temp_c,annual_max_temp_c,annual_min_temp_c,annual_precipitation_mm,annual_et0_mm,hot_days,dry_days,heavy_rain_days,monsoon_mean_temp_c,monsoon_precipitation_mm,monsoon_et0_mm,annual_water_balance_mm,monsoon_water_balance_mm,latitude,longitude
0,1,Andhra Pradesh,54,Chittoor,1966,26.586849,41.9,12.6,1266.9,1702.26,91,215,2,27.266393,710.1,526.48,-435.36,183.62,13.21055,79.0956
1,1,Andhra Pradesh,54,Chittoor,1967,26.746027,40.0,13.3,847.3,1784.34,78,239,2,28.562295,317.9,606.82,-937.04,-288.92,13.21055,79.0956
2,1,Andhra Pradesh,54,Chittoor,1968,27.010383,39.6,14.0,752.1,1820.50,90,244,0,28.982787,375.5,668.47,-1068.40,-292.97,13.21055,79.0956
3,1,Andhra Pradesh,54,Chittoor,1969,27.544110,40.6,14.2,805.1,1861.22,131,254,1,29.336066,233.5,700.06,-1056.12,-466.56,13.21055,79.0956
4,1,Andhra Pradesh,54,Chittoor,1970,26.816164,40.0,12.5,643.4,1776.33,72,250,0,28.354918,299.5,617.38,-1132.93,-317.88,13.21055,79.0956


FULL-SPAN WEATHER PILOT VERIFIED


In [ ]:
from pathlib import Path
import gzip
import hashlib
import json
import time
import requests
import pandas as pd

base = Path("/content/drive/MyDrive/FarmSight_XAI")

coords_path = base / "data/interim/district_coordinates_accepted.csv"
agri_path = base / "data/interim/icrisat_rice_maize_cotton_long.csv"

raw_weather_dir = base / "data/raw/era5_daily_by_district"
district_feature_dir = base / "data/interim/era5_annual_by_district"

combined_output = (
    base / "data/interim/era5_district_year_weather_features.csv"
)
manifest_output = base / "logs/weather_acquisition_manifest.csv"
report_output = base / "logs/weather_full_acquisition_report.txt"

raw_weather_dir.mkdir(parents=True, exist_ok=True)
district_feature_dir.mkdir(parents=True, exist_ok=True)

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as file:
        for block in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

coords_hash_before = sha256_file(coords_path)
agri_hash_before = sha256_file(agri_path)

coords = pd.read_csv(coords_path)
agri = pd.read_csv(agri_path)

lat_col = next(
    c for c in coords.columns
    if "latitude" in c.lower() or c.lower() == "lat"
)
lon_col = next(
    c for c in coords.columns
    if "longitude" in c.lower() or c.lower() in {"lon", "lng"}
)

start_year = int(agri["year"].min())
end_year = int(agri["year"].max())
expected_years = end_year - start_year + 1

session = requests.Session()
manifest_rows = []

def create_annual_features(payload, row):
    daily = pd.DataFrame(payload["daily"])

    daily["date"] = pd.to_datetime(daily["time"])
    daily["year"] = daily["date"].dt.year
    daily["month"] = daily["date"].dt.month

    daily["hot_day"] = daily["temperature_2m_max"] >= 35
    daily["dry_day"] = daily["precipitation_sum"] < 1
    daily["heavy_rain_day"] = daily["precipitation_sum"] >= 50

    annual = daily.groupby("year").agg(
        annual_mean_temp_c=("temperature_2m_mean", "mean"),
        annual_max_temp_c=("temperature_2m_max", "max"),
        annual_min_temp_c=("temperature_2m_min", "min"),
        annual_precipitation_mm=("precipitation_sum", "sum"),
        annual_et0_mm=("et0_fao_evapotranspiration", "sum"),
        hot_days=("hot_day", "sum"),
        dry_days=("dry_day", "sum"),
        heavy_rain_days=("heavy_rain_day", "sum")
    ).reset_index()

    monsoon = (
        daily[daily["month"].between(6, 9)]
        .groupby("year")
        .agg(
            monsoon_mean_temp_c=("temperature_2m_mean", "mean"),
            monsoon_precipitation_mm=("precipitation_sum", "sum"),
            monsoon_et0_mm=("et0_fao_evapotranspiration", "sum")
        )
        .reset_index()
    )

    annual = annual.merge(monsoon, on="year", how="left")

    annual["annual_water_balance_mm"] = (
        annual["annual_precipitation_mm"] -
        annual["annual_et0_mm"]
    )

    annual["monsoon_water_balance_mm"] = (
        annual["monsoon_precipitation_mm"] -
        annual["monsoon_et0_mm"]
    )

    annual.insert(0, "district_name", row["district_name"])
    annual.insert(0, "district_code", int(row["district_code"]))
    annual.insert(0, "state_name", row["state_name"])
    annual.insert(0, "state_code", int(row["state_code"]))

    annual["latitude"] = float(row[lat_col])
    annual["longitude"] = float(row[lon_col])

    return annual

for position, (_, row) in enumerate(coords.iterrows(), start=1):
    state_code = int(row["state_code"])
    district_code = int(row["district_code"])

    file_id = f"state_{state_code}_district_{district_code}"

    raw_path = raw_weather_dir / f"{file_id}.json.gz"
    annual_path = district_feature_dir / f"{file_id}.csv"

    # Resume safely if this district has already been verified.
    if annual_path.exists():
        existing = pd.read_csv(annual_path)

        if (
            len(existing) == expected_years
            and existing["year"].nunique() == expected_years
        ):
            manifest_rows.append({
                "state_code": state_code,
                "district_code": district_code,
                "district_name": row["district_name"],
                "status": "existing_verified",
                "attempts": 0,
                "elapsed_seconds": 0,
                "error": ""
            })

            print(
                f"[{position}/{len(coords)}] "
                f"{row['district_name']}: already verified"
            )
            continue

    params = {
        "latitude": float(row[lat_col]),
        "longitude": float(row[lon_col]),
        "start_date": f"{start_year}-01-01",
        "end_date": f"{end_year}-12-31",
        "daily": [
            "temperature_2m_mean",
            "temperature_2m_max",
            "temperature_2m_min",
            "precipitation_sum",
            "et0_fao_evapotranspiration"
        ],
        "timezone": "Asia/Kolkata",
        "models": "era5"
    }

    success = False
    last_error = ""
    request_start = time.time()

    for attempt in range(1, 5):
        try:
            response = session.get(
                "https://archive-api.open-meteo.com/v1/archive",
                params=params,
                timeout=300
            )
            response.raise_for_status()

            payload = response.json()
            annual = create_annual_features(payload, row)

            feature_columns = [
                c for c in annual.columns
                if c not in {
                    "state_code", "state_name",
                    "district_code", "district_name",
                    "year", "latitude", "longitude"
                }
            ]

            if len(annual) != expected_years:
                raise ValueError(
                    f"Expected {expected_years} years, got {len(annual)}"
                )

            if annual[feature_columns].isna().any().any():
                raise ValueError("Missing annual weather features detected")

            with gzip.open(raw_path, "wt", encoding="utf-8") as file:
                json.dump(payload, file)

            annual.to_csv(annual_path, index=False)

            success = True
            break

        except Exception as error:
            last_error = str(error)
            wait_seconds = 5 * (2 ** (attempt - 1))

            print(
                f"[{position}/{len(coords)}] "
                f"{row['district_name']} attempt {attempt} failed: "
                f"{last_error}"
            )

            if attempt < 4:
                time.sleep(wait_seconds)

    elapsed = round(time.time() - request_start, 2)

    manifest_rows.append({
        "state_code": state_code,
        "district_code": district_code,
        "district_name": row["district_name"],
        "status": "downloaded_verified" if success else "failed",
        "attempts": attempt,
        "elapsed_seconds": elapsed,
        "error": "" if success else last_error
    })

    pd.DataFrame(manifest_rows).to_csv(
        manifest_output, index=False
    )

    print(
        f"[{position}/{len(coords)}] {row['district_name']}: "
        f"{'VERIFIED' if success else 'FAILED'}"
    )

    # Polite delay for the public API.
    time.sleep(1)

# Combine every verified district file.
annual_files = sorted(district_feature_dir.glob("*.csv"))
combined = pd.concat(
    [pd.read_csv(path) for path in annual_files],
    ignore_index=True
)

combined.to_csv(combined_output, index=False)

manifest = pd.DataFrame(manifest_rows)
failed_count = int((manifest["status"] == "failed").sum())

duplicate_keys = int(
    combined.duplicated(
        ["state_code", "district_code", "year"]
    ).sum()
)

missing_values = int(combined.isna().sum().sum())
expected_rows = len(coords) * expected_years

source_hashes_unchanged = (
    sha256_file(coords_path) == coords_hash_before
    and sha256_file(agri_path) == agri_hash_before
)

verified = (
    failed_count == 0
    and len(annual_files) == len(coords)
    and len(combined) == expected_rows
    and duplicate_keys == 0
    and missing_values == 0
    and source_hashes_unchanged
)

report = f"""Full ERA5 Weather Acquisition
Districts expected: {len(coords)}
District files created: {len(annual_files)}
Period: {start_year}-{end_year}
Years per district: {expected_years}
Expected combined rows: {expected_rows}
Actual combined rows: {len(combined)}
Failed districts: {failed_count}
Duplicate district-year keys: {duplicate_keys}
Missing values: {missing_values}
Source hashes unchanged: {source_hashes_unchanged}
Acquisition verified: {verified}
"""

report_output.write_text(report, encoding="utf-8")

print("\n" + report)

if verified:
    print("FULL ERA5 WEATHER ACQUISITION VERIFIED")
else:
    print("WEATHER ACQUISITION INCOMPLETE — RERUN THIS CELL TO RESUME")

[1/229] Chittoor: VERIFIED
[2/229] Guntur attempt 1 failed: 429 Client Error: Too Many Requests for url: https://archive-api.open-meteo.com/v1/archive?latitude=16.29974&longitude=80.45729&start_date=1966-01-01&end_date=2017-12-31&daily=temperature_2m_mean&daily=temperature_2m_max&daily=temperature_2m_min&daily=precipitation_sum&daily=et0_fao_evapotranspiration&timezone=Asia%2FKolkata&models=era5
[2/229] Guntur: VERIFIED
[3/229] Kurnool attempt 1 failed: 429 Client Error: Too Many Requests for url: https://archive-api.open-meteo.com/v1/archive?latitude=15.82887&longitude=78.03602&start_date=1966-01-01&end_date=2017-12-31&daily=temperature_2m_mean&daily=temperature_2m_max&daily=temperature_2m_min&daily=precipitation_sum&daily=et0_fao_evapotranspiration&timezone=Asia%2FKolkata&models=era5
[3/229] Kurnool attempt 2 failed: 429 Client Error: Too Many Requests for url: https://archive-api.open-meteo.com/v1/archive?latitude=15.82887&longitude=78.03602&start_date=1966-01-01&end_date=2017-12-31

KeyboardInterrupt: 

In [ ]:
from pathlib import Path

folder = Path(
    "/content/drive/MyDrive/FarmSight_XAI/"
    "data/interim/era5_annual_by_district"
)

files = sorted(folder.glob("*.csv"))
print("Saved district files:", len(files))

for file in files:
    print(file.name)

Saved district files: 6
state_17_district_602.csv
state_17_district_607.csv
state_17_district_610.csv
state_1_district_44.csv
state_1_district_49.csv
state_1_district_54.csv


In [ ]:
if "response" in globals():
    print("Last HTTP status:", response.status_code)
    print("Retry-After:", response.headers.get("Retry-After", "Not supplied"))
    print("Server message:", response.text[:1500])
else:
    print("Previous response unavailable. No request made.")

Last HTTP status: 429
Retry-After: Not supplied
Server message: {"error":true,"reason":"Minutely API request limit exceeded. Please try again in one minute."}


In [ ]:
import gzip
import json
import time
import requests
import pandas as pd

# Reuse the setup and feature function from cell 33.
required = [
    "coords", "district_feature_dir", "raw_weather_dir",
    "create_annual_features", "start_year", "end_year",
    "lat_col", "lon_col"
]
assert all(name in globals() for name in required), (
    "Setup unavailable. Stop and share this error; do not rerun cell 33."
)

pending = []
for _, row in coords.iterrows():
    file_id = (
        f"state_{int(row['state_code'])}_"
        f"district_{int(row['district_code'])}"
    )
    if not (district_feature_dir / f"{file_id}.csv").exists():
        pending.append((file_id, row))

print("Remaining districts:", len(pending))
completed = 0

for file_id, row in pending[:20]:
    print("Waiting 65 seconds before:", row["district_name"], flush=True)
    time.sleep(65)

    params = {
        "latitude": float(row[lat_col]),
        "longitude": float(row[lon_col]),
        "start_date": f"{start_year}-01-01",
        "end_date": f"{end_year}-12-31",
        "daily": [
            "temperature_2m_mean", "temperature_2m_max",
            "temperature_2m_min", "precipitation_sum",
            "et0_fao_evapotranspiration"
        ],
        "timezone": "Asia/Kolkata",
        "models": "era5"
    }

    response = requests.get(
        "https://archive-api.open-meteo.com/v1/archive",
        params=params,
        timeout=300
    )

    if response.status_code == 429:
        print("RATE LIMITED — batch stopped:", response.text)
        break

    response.raise_for_status()
    payload = response.json()
    daily = pd.DataFrame(payload["daily"])

    expected_dates = pd.date_range(
        f"{start_year}-01-01", f"{end_year}-12-31"
    )
    actual_dates = pd.DatetimeIndex(pd.to_datetime(daily["time"]))

    assert actual_dates.equals(expected_dates), "Incomplete/incorrect dates"
    assert daily.notna().all().all(), "Missing daily weather values"

    annual = create_annual_features(payload, row)
    assert len(annual) == end_year - start_year + 1
    assert annual.notna().all().all(), "Missing annual values"

    with gzip.open(
        raw_weather_dir / f"{file_id}.json.gz", "wt", encoding="utf-8"
    ) as file:
        json.dump(payload, file)

    annual.to_csv(
        district_feature_dir / f"{file_id}.csv", index=False
    )
    completed += 1
    print("Saved and checked:", row["district_name"], flush=True)

print("New districts completed:", completed)
print(
    "Total saved district files:",
    len(list(district_feature_dir.glob("*.csv")))
)

Remaining districts: 220
Waiting 65 seconds before: Nagaon
Saved and checked: Nagaon
Waiting 65 seconds before: Bhagalpur
Saved and checked: Bhagalpur
Waiting 65 seconds before: Darbhanga
Saved and checked: Darbhanga
Waiting 65 seconds before: Gaya
Saved and checked: Gaya
Waiting 65 seconds before: Muzaffarpur
Saved and checked: Muzaffarpur
Waiting 65 seconds before: Patna
RATE LIMITED — batch stopped: {"reason":"Daily API request limit exceeded. Please try again tomorrow.","error":true}
New districts completed: 5
Total saved district files: 14


In [ ]:
from google.colab import userdata

cds_token = userdata.get("CDS_API_KEY").strip()
assert cds_token, "The secret is empty."
assert not cds_token.startswith("key:"), "Store only the token, without 'key:'."

print("CDS secret accessible. Token kept hidden.")

CDS secret accessible. Token kept hidden.


In [ ]:
%pip -q install "cdsapi>=0.7.7"

import cdsapi
from google.colab import userdata

cds_client = cdsapi.Client(
    url="https://cds.climate.copernicus.eu/api",
    key=userdata.get("CDS_API_KEY").strip(),
    debug=False
)

print("CDS client ready. Authentication still needs a pilot download.")

2026-10-16 13:57:35,950 INFO [2026-10-06T00:00:00Z] The [CDS Terms of Use](https://cds.climate.copernicus.eu/disclaimer-privacy) will be updated on 28 October 2026. Continued use of Copernicus Products and Services through the CDS after this date constitutes acceptance of the updated terms. If you do not accept the change, please [contact ECMWF Support](https://jira.ecmwf.int/plugins/servlet/desk/portal/1/create/202) before 27 October 2026.
 **[More information about the upcoming change.](https://forum.ecmwf.int/t/upcoming-cds-terms-of-use-change/15374)**
INFO:ecmwf.datastores.legacy_client:[2026-10-06T00:00:00Z] The [CDS Terms of Use](https://cds.climate.copernicus.eu/disclaimer-privacy) will be updated on 28 October 2026. Continued use of Copernicus Products and Services through the CDS after this date constitutes acceptance of the updated terms. If you do not accept the change, please [contact ECMWF Support](https://jira.ecmwf.int/plugins/servlet/desk/portal/1/create/202) before 27 

CDS client ready. Authentication still needs a pilot download.


In [ ]:
%pip -q install xarray netCDF4

from pathlib import Path
import pandas as pd
import xarray as xr

base = Path("/content/drive/MyDrive/FarmSight_XAI")
assert base.is_dir(), "Google Drive project folder unavailable."

coords = pd.read_csv(
    base / "data/interim/district_coordinates_accepted.csv"
)
sample = coords.iloc[0]

output = base / "data/raw/cds_era5_pilot_20010101.nc"

request = {
    "variable": ["2m_temperature", "total_precipitation"],
    "location": {
        "longitude": float(sample["longitude"]),
        "latitude": float(sample["latitude"])
    },
    "date": ["2001-01-01/2001-01-01"],
    "data_format": "netcdf"
}

print("Requesting district:", sample["district_name"], flush=True)

cds_client.retrieve(
    "reanalysis-era5-single-levels-timeseries",
    request,
    str(output)
)

with xr.open_dataset(output) as ds:
    print(ds)

    assert "t2m" in ds and "tp" in ds, "Expected variables missing."
    assert ds.sizes.get("valid_time") == 24, "Expected 24 hourly records."

    times = pd.DatetimeIndex(ds["valid_time"].values)
    expected = pd.date_range("2001-01-01", periods=24, freq="h")
    assert times.equals(expected), "Unexpected timestamps."
    assert bool(ds["t2m"].notnull().all()), "Missing temperatures."
    assert bool(ds["tp"].notnull().all()), "Missing precipitation."

    print("Temperature units:", ds["t2m"].attrs.get("units"))
    print("Precipitation units:", ds["tp"].attrs.get("units"))

print("Saved:", output.name)
print("File size:", output.stat().st_size, "bytes")
print("CDS PILOT DOWNLOAD VERIFIED")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 6.2 MB/s eta 0:00:00
Requesting district: Chittoor


2026-10-16 14:00:49,023 INFO Request ID is 78955e85-b91a-4868-91ab-98b9f95ffa5c
INFO:ecmwf.datastores.legacy_client:Request ID is 78955e85-b91a-4868-91ab-98b9f95ffa5c
2026-10-16 14:00:49,719 INFO status has been updated to accepted
INFO:ecmwf.datastores.legacy_client:status has been updated to accepted
2026-10-16 14:01:40,425 INFO status has been updated to running
INFO:ecmwf.datastores.legacy_client:status has been updated to running
2026-10-16 14:02:06,229 INFO status has been updated to successful
INFO:ecmwf.datastores.legacy_client:status has been updated to successful


db64a036fe1f68bac9ba96b2d38717fe.zip:   0%|          | 0.00/20.4k [00:00<?, ?B/s]

ValueError: did not find a match in any of xarray's currently installed IO backends ['netcdf4', 'h5netcdf', 'scipy', 'ee']. Consider explicitly selecting one of the installed engines via the ``engine`` parameter, or installing additional IO dependencies, see:
https://docs.xarray.dev/en/stable/getting-started-guide/installing.html
https://docs.xarray.dev/en/stable/user-guide/io.html

In [ ]:
import os
from pathlib import Path
import zipfile
import xarray as xr
import pandas as pd

base = Path("/content/drive/MyDrive/FarmSight_XAI")
pilot_file = base / "data/raw/cds_era5_pilot_20010101.nc"
extract_dir = base / "data/raw/cds_pilot_extracted"
report_path = base / "logs/cds_pilot_validation.txt"

# 1. Check existing file
assert pilot_file.is_file(), f"File not found at {pilot_file}"
file_size = pilot_file.stat().st_size
print(f"File exists: {pilot_file.name}")
print(f"File size: {file_size} bytes")

# 2. Inspect binary signature (magic bytes)
with open(pilot_file, "rb") as f:
    magic = f.read(4)
print(f"Magic bytes: {magic}")

is_zip = magic.startswith(b"PK\x03\x04")
print(f"Is ZIP archive: {is_zip}")

File exists: cds_era5_pilot_20010101.nc
File size: 20894 bytes
Magic bytes: b'PK\x03\x04'
Is ZIP archive: True


In [ ]:
extracted_files = []
if is_zip:
    print("Extracting ZIP members safely...")
    extract_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(pilot_file, 'r') as z:
        for member in z.namelist():
            # Resolve absolute paths and avoid directory traversal vulnerabilities
            resolved_path = (extract_dir / member).resolve()
            if not resolved_path.is_relative_to(extract_dir.resolve()):
                print(f"Skipping unsafe member: {member}")
                continue
            z.extract(member, path=extract_dir)
            extracted_files.append(resolved_path)
            print(f"Extracted: {member}")
else:
    extracted_files = [pilot_file]
    print("No ZIP extraction needed. Processing file directly.")

Extracting ZIP members safely...
Extracted: reanalysis-era5-single-levels-timeseries-sfcsxju9bl9.nc


In [ ]:
# Install necessary engines if reading NetCDF4 fails or if we need specific backends
%pip -q install scipy netCDF4 h5netcdf

In [ ]:
# 4. & 5. Open files and inspect variables
datasets = []
for file_path in extracted_files:
    try:
        print(f"\nOpening: {file_path.name}")
        ds = xr.open_dataset(file_path)
        print(ds)
        datasets.append(ds)
    except Exception as e:
        print(f"Failed to open {file_path.name} with default engines: {e}")
        # Attempt with explicit engine if needed
        try:
            ds = xr.open_dataset(file_path, engine='scipy')
            print("Opened successfully with scipy engine.")
            datasets.append(ds)
        except Exception as e2:
            print(f"Failed with scipy: {e2}")


Opening: reanalysis-era5-single-levels-timeseries-sfcsxju9bl9.nc
<xarray.Dataset> Size: 400B
Dimensions:     (valid_time: 24)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 192B 2001-01-01 ... 2001-01-01T23...
    latitude    float64 8B ...
    longitude   float64 8B ...
Data variables:
    t2m         (valid_time) float32 96B ...
    tp          (valid_time) float32 96B ...
Attributes:
    Conventions:             CF-1.7
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_edition:            1
    GRIB_subCentre:          0
    history:                 2024-09-02T04:48 GRIB to CDM+CF via cfgrib-0.9.1...
    institution:             European Centre for Medium-Range Weather Forecasts


In [ ]:
# 6. Validate timestamps, variables, and missing values
all_valid = True
problems = []

if not datasets:
    all_valid = False
    problems.append("No datasets could be successfully loaded from the pilot file.")
else:
    # Combine if multiple datasets exist, otherwise use the single one
    if len(datasets) == 1:
        ds = datasets[0]
    else:
        ds = xr.merge(datasets)

    # Validate expected variables (temperature and precipitation)
    # Variable names in new CDS single levels are typically 't2m' / 'tp' or similar
    vars_found = list(ds.data_vars)
    print("Variables found:", vars_found)

    # Checking for typical hourly count
    time_dim = 'valid_time' if 'valid_time' in ds.dims else ('time' if 'time' in ds.dims else None)
    if not time_dim:
        all_valid = False
        problems.append("Time dimension not identified in the dataset.")
    else:
        num_times = ds.sizes[time_dim]
        print(f"Time dimension: {time_dim} with {num_times} stamps.")

        times = pd.DatetimeIndex(ds[time_dim].values)
        print("Timestamps index:")
        print(times)

        # Expected 24 hourly values on 2001-01-01
        expected_range = pd.date_range("2001-01-01 00:00:00", "2001-01-01 23:00:00", freq="h", tz="UTC")
        # Standardize timezone comparison if tz is present in dataset
        if times.tz is None and expected_range.tz is not None:
            expected_range = expected_range.tz_localize(None)

        if not times.equals(expected_range):
            all_valid = False
            problems.append(f"Timestamps do not match expected 24 hourly sequence. Unique timestamps: {len(times.unique())}")

        # Check for missing values
        for var in vars_found:
            missing_count = int(ds[var].isnull().sum().values)
            print(f"Variable '{var}' units: {ds[var].attrs.get('units')}, Missing values count: {missing_count}")
            if missing_count > 0:
                all_valid = False
                problems.append(f"Variable '{var}' contains {missing_count} missing values.")

Variables found: ['t2m', 'tp']
Time dimension: valid_time with 24 stamps.
Timestamps index:
DatetimeIndex(['2001-01-01 00:00:00', '2001-01-01 01:00:00',
               '2001-01-01 02:00:00', '2001-01-01 03:00:00',
               '2001-01-01 04:00:00', '2001-01-01 05:00:00',
               '2001-01-01 06:00:00', '2001-01-01 07:00:00',
               '2001-01-01 08:00:00', '2001-01-01 09:00:00',
               '2001-01-01 10:00:00', '2001-01-01 11:00:00',
               '2001-01-01 12:00:00', '2001-01-01 13:00:00',
               '2001-01-01 14:00:00', '2001-01-01 15:00:00',
               '2001-01-01 16:00:00', '2001-01-01 17:00:00',
               '2001-01-01 18:00:00', '2001-01-01 19:00:00',
               '2001-01-01 20:00:00', '2001-01-01 21:00:00',
               '2001-01-01 22:00:00', '2001-01-01 23:00:00'],
              dtype='datetime64[ns]', freq=None)
Variable 't2m' units: K, Missing values count: 0
Variable 'tp' units: m, Missing values count: 0


In [ ]:
# 7. Save report
report_path.parent.mkdir(parents=True, exist_ok=True)

validation_status = "SUCCESS" if all_valid else "FAILED"
problems_desc = "\n".join(f"- {p}" for p in problems) if problems else "None detected."

report_text = f"""CDS Pilot Dataset Validation Report
==================================
File Name: {pilot_file.name}
File Size: {file_size} bytes
Format (is ZIP): {is_zip}

Validation Status: {validation_status}

Problems Identified:
{problems_desc}

Dataset variables discovered: {vars_found if 'vars_found' in locals() else 'N/A'}
Timestamps validated: {all_valid}
"""

with open(report_path, "w", encoding="utf-8") as f:
    f.write(report_text)

print(f"Report saved to: {report_path}")

Report saved to: /content/drive/MyDrive/FarmSight_XAI/logs/cds_pilot_validation.txt


In [ ]:
# 9. Final Validation Verification output
if all_valid:
    print("CDS PILOT VERIFIED")
else:
    print(f"CDS PILOT FAILED: {problems_desc}")

CDS PILOT VERIFIED


In [ ]:
import os
import time
import zipfile
import hashlib
from pathlib import Path
import pandas as pd
import xarray as xr
from google.colab import userdata
import cdsapi

base = Path("/content/drive/MyDrive/FarmSight_XAI")
coords_path = base / "data/interim/district_coordinates_accepted.csv"
output_dir = base / "data/raw/cds_fullspan_pilot"
output_dir.mkdir(parents=True, exist_ok=True)

log_path = base / "logs/cds_fullspan_pilot_validation.txt"

# 1. Load coordinates and select Chittoor (first district)
coords = pd.read_csv(coords_path)
sample = coords.iloc[0]

# Configure Target Path
target_file = output_dir / "cds_era5_fullspan_pilot_1966_2017.nc"
extract_dir = output_dir / "extracted"
extract_dir.mkdir(parents=True, exist_ok=True)

start_year = 1966
end_year = 2017

request = {
    "variable": ["2m_temperature", "total_precipitation"],
    "location": {
        "longitude": float(sample["longitude"]),
        "latitude": float(sample["latitude"])
    },
    "date": [f"{start_year}-01-01/{end_year}-12-31"],
    "data_format": "netcdf"
}

# 2. Check if already exists to avoid redundant downloads
download_needed = True
elapsed_time = 0.0

if target_file.is_file() and target_file.stat().st_size > 0:
    print(f"Existing download found at {target_file}. Skipping API request.")
    download_needed = False
else:
    print(f"Requesting full-span data for {sample['district_name']} ({start_year}-{end_year})...", flush=True)
    cds_client = cdsapi.Client(
        url="https://cds.climate.copernicus.eu/api",
        key=userdata.get("CDS_API_KEY").strip(),
        debug=False
    )
    start_time = time.time()
    cds_client.retrieve(
        "reanalysis-era5-single-levels-timeseries",
        request,
        str(target_file)
    )
    elapsed_time = round(time.time() - start_time, 2)
    print(f"Download completed in {elapsed_time} seconds.")


Requesting full-span data for Chittoor (1966-2017)...


2026-10-16 15:19:12,334 INFO [2026-10-06T00:00:00Z] The [CDS Terms of Use](https://cds.climate.copernicus.eu/disclaimer-privacy) will be updated on 28 October 2026. Continued use of Copernicus Products and Services through the CDS after this date constitutes acceptance of the updated terms. If you do not accept the change, please [contact ECMWF Support](https://jira.ecmwf.int/plugins/servlet/desk/portal/1/create/202) before 27 October 2026.
 **[More information about the upcoming change.](https://forum.ecmwf.int/t/upcoming-cds-terms-of-use-change/15374)**
INFO:ecmwf.datastores.legacy_client:[2026-10-06T00:00:00Z] The [CDS Terms of Use](https://cds.climate.copernicus.eu/disclaimer-privacy) will be updated on 28 October 2026. Continued use of Copernicus Products and Services through the CDS after this date constitutes acceptance of the updated terms. If you do not accept the change, please [contact ECMWF Support](https://jira.ecmwf.int/plugins/servlet/desk/portal/1/create/202) before 27 

66d24688517ed2d7a0a24f85cb4ce275.zip:   0%|          | 0.00/6.98M [00:00<?, ?B/s]

Download completed in 56.34 seconds.


In [ ]:
# 3. Format Detection & ZIP Extraction
with open(target_file, "rb") as f:
    magic = f.read(4)
is_zip = magic.startswith(b"PK\x03\x04")
print(f"Detected file format: {'ZIP archive' if is_zip else 'NetCDF'}")

extracted_nc_files = []
if is_zip:
    print("Unpacking ZIP safely...")
    with zipfile.ZipFile(target_file, 'r') as z:
        for member in z.namelist():
            resolved_path = (extract_dir / member).resolve()
            if not resolved_path.is_relative_to(extract_dir.resolve()):
                print(f"Skipping unsafe member: {member}")
                continue
            z.extract(member, path=extract_dir)
            extracted_nc_files.append(resolved_path)
            print(f"Extracted member: {member}")
else:
    extracted_nc_files = [target_file]

# Load dataset using xarray
if len(extracted_nc_files) == 1:
    ds_full = xr.open_dataset(extracted_nc_files[0])
else:
    ds_full = xr.open_mfdataset([str(p) for p in extracted_nc_files])

print("Dataset loaded successfully:")
print(ds_full)


Detected file format: ZIP archive
Unpacking ZIP safely...
Extracted member: reanalysis-era5-single-levels-timeseries-sfcm1vchjse.nc
Dataset loaded successfully:
<xarray.Dataset> Size: 7MB
Dimensions:     (valid_time: 455832)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 4MB 1966-01-01 ... 2017-12-31T23:...
    latitude    float64 8B ...
    longitude   float64 8B ...
Data variables:
    t2m         (valid_time) float32 2MB ...
    tp          (valid_time) float32 2MB ...
Attributes:
    Conventions:             CF-1.7
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_edition:            1
    GRIB_subCentre:          0
    history:                 2024-09-02T04:48 GRIB to CDM+CF via cfgrib-0.9.1...
    institution:             European Centre for Medium-Range Weather Forecasts


In [ ]:
# 4. Validation of Variables & Temporal Completeness
expected_index = pd.date_range(
    start=f"{start_year}-01-01 00:00:00",
    end=f"{end_year}-12-31 23:00:00",
    freq="h",
    tz="UTC"
)

time_dim = 'valid_time' if 'valid_time' in ds_full.dims else ('time' if 'time' in ds_full.dims else None)
assert time_dim is not None, "Time dimension not found in dataset."

actual_times = pd.DatetimeIndex(ds_full[time_dim].values)
if actual_times.tz is None:
    actual_times = actual_times.tz_localize("UTC")

timestamps_match = actual_times.equals(expected_index)
expected_hours = len(expected_index)
actual_hours = len(actual_times)

t2m_missing = int(ds_full["t2m"].isnull().sum().values)
tp_missing = int(ds_full["tp"].isnull().sum().values)
t2m_units = ds_full["t2m"].attrs.get("units", "N/A")
tp_units = ds_full["tp"].attrs.get("units", "N/A")

checks_passed = (
    timestamps_match
    and t2m_missing == 0
    and tp_missing == 0
)

# Determine coordinates
grid_lat = float(ds_full["latitude"].values) if "latitude" in ds_full.coords else float(sample["latitude"])
grid_lon = float(ds_full["longitude"].values) if "longitude" in ds_full.coords else float(sample["longitude"])

report_text = f"""CDS Full-Span Feasibility Pilot Report
======================================
District: {sample['district_name']}
State: {sample['state_name']}
Target Coordinates (Query): Lat {sample['latitude']}, Lon {sample['longitude']}
Grid Coordinates (Returned): Lat {grid_lat}, Lon {grid_lon}
File Name: {target_file.name}
File Size: {target_file.stat().st_size} bytes
Format (Is ZIP): {is_zip}

Temporal Audit:
---------------
Target Period: {start_year}-01-01 00:00 UTC to {end_year}-12-31 23:00 UTC
Expected Hours: {expected_hours}
Actual Hours: {actual_hours}
Timestamps Match Exactly: {timestamps_match}

Variable Quality Controls:
--------------------------
2m Temperature ('t2m'):
  - Units: {t2m_units}
  - Missing Values: {t2m_missing}
Total Precipitation ('tp'):
  - Units: {tp_units}
  - Missing Values: {tp_missing}

Validation Status: {'SUCCESS' if checks_passed else 'FAILED'}
"""

with open(log_path, "w", encoding="utf-8") as f:
    f.write(report_text)

print(report_text)

if checks_passed:
    print("CDS FULL-SPAN PILOT VERIFIED")
else:
    print("CDS FULL-SPAN PILOT VALIDATION FAILED")


CDS Full-Span Feasibility Pilot Report
District: Chittoor
State: Andhra Pradesh
Target Coordinates (Query): Lat 13.21055, Lon 79.0956
Grid Coordinates (Returned): Lat 13.25, Lon 79.0
File Name: cds_era5_fullspan_pilot_1966_2017.nc
File Size: 7315321 bytes
Format (Is ZIP): True

Temporal Audit:
---------------
Target Period: 1966-01-01 00:00 UTC to 2017-12-31 23:00 UTC
Expected Hours: 455832
Actual Hours: 455832
Timestamps Match Exactly: True

Variable Quality Controls:
--------------------------
2m Temperature ('t2m'):
  - Units: K
  - Missing Values: 0
Total Precipitation ('tp'):
  - Units: m
  - Missing Values: 0

Validation Status: SUCCESS

CDS FULL-SPAN PILOT VERIFIED
